In [1]:
import os
import pandas as pd
import requests
import time

In [4]:
# Path to the CSV file
csv_file_path = 'file_urls.csv'  # Replace with your CSV file path

# Folder to save downloaded files
download_folder = 'downloads'

# Create the downloads folder if it does not exist
if not os.path.exists(download_folder):
    os.makedirs(download_folder)

# Read the CSV file
try:
    df = pd.read_csv(csv_file_path)
except FileNotFoundError:
    print(f"The file '{csv_file_path}' does not exist.")
    exit()

# Assuming the CSV file has a column named 'url' that contains the file URLs
if 'url' not in df.columns:
    print("The CSV file does not contain a column named 'url'.")
    exit()

# Function to get the filename from the response headers or URL
def get_filename_from_response(response, url):
    # Try to get the filename from the Content-Disposition header
    if 'Content-Disposition' in response.headers:
        content_disposition = response.headers['Content-Disposition']
        filename = content_disposition.split('filename=')[-1].strip('\"')
    else:
        # Fallback to the filename in the URL
        filename = url.split('/')[-1]
    return filename

# Function to download a file from a URL
def download_file(url, folder):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Check for errors

        # Get the filename from the response headers or URL
        filename = get_filename_from_response(response, url)
        filepath = os.path.join(folder, filename)

        # Write the file to the specified folder
        with open(filepath, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        print(f"Downloaded: {filepath}")
    except requests.exceptions.RequestException as e:
        print(f"Failed to download {url}: {e}")

In [5]:
# Download each file in the CSV
for url in df['url']:
    download_file(url, download_folder)
    time.sleep(0.05)  # Pause for 50 milliseconds

Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Himanshu_Rohang.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Ronit_Rohang.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Ranbir_Gurung.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Mandeep_Kashyap.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Kushal_Rout.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Mirjang_Rongpee.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Riyan_chanda.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Abdar_Ali.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Gaurav_kumar.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Bristi_Rani_Das.pdf
Downloaded: downloads/Report_Card_Formative_Assessment_-_III_2025-2026_V_Abhilasha_Das.pdf


In [6]:
import os
import shutil
from PyPDF2 import PdfMerger

downloaded_folder = "downloads"
destination_root = "class-wise"

# Step 1: Copy and organize PDF files by Exam and Class
for filename in os.listdir(downloaded_folder):
    if filename.endswith(".pdf") and "Report_Card" in filename:
        parts = filename.split("_")
        try:
            # Extract exam and class info from filename
            exam_start_index = 2
            exam_end_index = next(i for i, part in enumerate(parts) if part.count("-") == 1 and len(part) == 9)
            exam_name = "_".join(parts[exam_start_index:exam_end_index + 1])
            class_name = parts[exam_end_index + 1]

            dest_folder = os.path.join(destination_root, exam_name, class_name)
            os.makedirs(dest_folder, exist_ok=True)

            src_path = os.path.join(downloaded_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            # ✅ Copy instead of move
            shutil.copy2(src_path, dest_path)
            print(f"Copied: {filename} → {dest_folder}/")

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")

# Step 2: Merge PDFs inside each class folder
for exam_folder in os.listdir(destination_root):
    exam_path = os.path.join(destination_root, exam_folder)
    if not os.path.isdir(exam_path):
        continue

    for class_folder in os.listdir(exam_path):
        class_path = os.path.join(exam_path, class_folder)
        if not os.path.isdir(class_path):
            continue

        merger = PdfMerger()
        pdf_files = sorted([f for f in os.listdir(class_path) if f.endswith(".pdf") and not f.startswith("Combined_")])
        if not pdf_files:
            continue

        for pdf in pdf_files:
            merger.append(os.path.join(class_path, pdf))

        merged_filename = f"Combined_{class_folder}.pdf"
        merged_path = os.path.join(exam_path, merged_filename)  # <-- changed here

        merger.write(merged_path)
        merger.close()

        print(f"✅ Merged {len(pdf_files)} PDFs → {merged_filename} (inside {exam_folder}/)")



Copied: Report_Card_Formative_Assessment_-_III_2025-2026_III_Niraj_pradhan.pdf → class-wise/Formative_Assessment_-_III_2025-2026/III/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_X_Jassika_Nunnem_Gupta.pdf → class-wise/Formative_Assessment_-_III_2025-2026/X/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_LKG_Deekshita_Baruah.pdf → class-wise/Formative_Assessment_-_III_2025-2026/LKG/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_II_Prerna_Chetri.pdf → class-wise/Formative_Assessment_-_III_2025-2026/II/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_VII_Pragyan_Bhuyan.pdf → class-wise/Formative_Assessment_-_III_2025-2026/VII/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_UKG_Arman_Ali.pdf → class-wise/Formative_Assessment_-_III_2025-2026/UKG/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_II_Jayesh_kr_Das.pdf → class-wise/Formative_Assessment_-_III_2025-2026/II/
Copied: Report_Card_Formative_Assessment_-_III_2025-2026_UKG_An